# ADMM TEST

In [ ]:
import time
import struct
import numpy as np
import scipy.sparse as sp
from pynq import Overlay, allocate, MMIO

# =====================================================================
# 1. Initialization and IP Setup
# =====================================================================
# IMPORTANT: Update this path to your actual bitstream location
bitstream_path = '/home/xilinx/admm/admm.bit' 
print(f"Loading overlay from {bitstream_path}...")
overlay = Overlay(bitstream_path)
print('Overlay loaded!')

CONTROL_ADDR   = 0xA0000000
CONTROL_R_ADDR = 0xA0010000
ADDR_RANGE     = 0x10000

print(f"Manually mapping Control   : {hex(CONTROL_ADDR)}")
print(f"Manually mapping Control_R : {hex(CONTROL_R_ADDR)}")

control_ip   = MMIO(CONTROL_ADDR, ADDR_RANGE)
control_r_ip = MMIO(CONTROL_R_ADDR, ADDR_RANGE)

In [ ]:
import os
import time
import struct
import numpy as np
import scipy.sparse as sp

from pynq import Overlay, allocate, MMIO

# =====================================================================
# 1. Initialization and IP Setup
# =====================================================================

bitstream_path = '/home/xilinx/admm/admm_fixed.bit'

print(f"Loading overlay from {bitstream_path}...")
overlay = Overlay(bitstream_path)
print("Overlay loaded!")

CONTROL_ADDR = 0xA0000000
CONTROL_R_ADDR = 0xA0010000
ADDR_RANGE = 0x10000

control_ip = MMIO(CONTROL_ADDR, ADDR_RANGE)
control_r_ip = MMIO(CONTROL_R_ADDR, ADDR_RANGE)

# =====================================================================
# 2. Load Canonicalized CVXPY QP Data
# =====================================================================

data_dir = "./data"

print(f"\nLoading QP data from: {os.path.abspath(data_dir)}")

# Load sparse matrices
P_sparse = sp.load_npz(os.path.join(data_dir, "P.npz")).tocsc()
A_sparse = sp.load_npz(os.path.join(data_dir, "A.npz")).tocsc()

# Load vectors
q = np.load(os.path.join(data_dir, "q.npy")).astype(np.float32)
l_in = np.load(os.path.join(data_dir, "l.npy")).astype(np.float32)
u_in = np.load(os.path.join(data_dir, "u.npy")).astype(np.float32)

# Problem dimensions
num_rows, num_cols = A_sparse.shape

print(f"Problem size: {num_rows} x {num_cols}")

# Extract original P diagonal for objective calculations later
P_diag_orig = P_sparse.diagonal().astype(np.float32)

# =====================================================================
# 2.5 Matrix Scaling (Scaling + Cost Scaling)
# =====================================================================

print("\nApplying Scaling...")

def apply_scaling(P_diag, A_sparse, q, l, u, iters=10):
    n = len(P_diag)
    m = A_sparse.shape[0]

    # Initialize scaling matrices
    E = np.ones(n, dtype=np.float32)
    D = np.ones(m, dtype=np.float32)

    P_scaled = P_diag.copy()
    A_scaled = A_sparse.copy()

    for _ in range(iters):
        A_abs = abs(A_scaled)

        # 1. Column norms
        A_col_norm = A_abs.max(axis=0).toarray().flatten()
        P_col_norm = np.abs(P_scaled)
        x_norm = np.maximum(P_col_norm, A_col_norm)
        x_norm = np.maximum(x_norm, 1e-4) # prevent division by zero

        # 2. Row norms
        z_norm = A_abs.max(axis=1).toarray().flatten()
        z_norm = np.maximum(z_norm, 1e-4)

        # 3. Update scaling factors
        E_new = 1.0 / np.sqrt(x_norm)
        D_new = 1.0 / np.sqrt(z_norm)
        E *= E_new
        D *= D_new

        # 4. Scale Matrices: P = E*P*E, A = D*A*E
        P_scaled = P_scaled * (E_new ** 2)
        A_scaled = sp.diags(D_new).dot(A_scaled).dot(sp.diags(E_new)).tocsc()

    # Scale Vectors
    q_scaled = q * E
    l_scaled = np.where(l == -np.inf, -np.inf, l * D)
    u_scaled = np.where(u ==  np.inf,  np.inf, u * D)

    # OSQP Cost scaling (c)
    c = 1.0 / np.maximum(np.max(np.abs(P_scaled)), np.max(np.abs(q_scaled)))
    c = max(c, 1e-4)
    P_scaled *= c
    q_scaled *= c

    return P_scaled, A_scaled, q_scaled, l_scaled, u_scaled, E, D, c

# Get scaled data and unscaling factors (E, D, c)
(P_diag_scaled, A_sparse_scaled, q_scaled, 
 l_scaled, u_scaled, E_scale, D_scale, c_scale) = apply_scaling(
    P_diag_orig, A_sparse, q, l_in, u_in
)

# =====================================================================
# 3. Extract CSC Data (USING SCALED DATA)
# =====================================================================

PACK_SIZE = 16

# A matrix
A_col_ptr = A_sparse_scaled.indptr.astype(np.int32)
A_row_idx = A_sparse_scaled.indices.astype(np.int32)
A_values = A_sparse_scaled.data.astype(np.float32)
A_nnz = int(A_sparse_scaled.nnz)

# A transpose
AT_sparse = A_sparse_scaled.transpose().tocsc()
AT_col_ptr = AT_sparse.indptr.astype(np.int32)
AT_row_idx = AT_sparse.indices.astype(np.int32)
AT_values = AT_sparse.data.astype(np.float32)

# P diagonal
P_values = P_diag_scaled.copy()
P_row_idx = np.arange(num_cols, dtype=np.int32)
P_col_ptr = np.arange(num_cols + 1, dtype=np.int32)
P_nnz = len(P_values)

print(f"A nnz: {A_nnz}")
print(f"P nnz: {P_nnz}")

# =====================================================================
# 4. Solver Parameters
# =====================================================================

alpha = 1.8
sigma = 1e-2

eps_abs = 1e-3
eps_rel = 1e-3

# Indirect solve accuracy control:
# - If MPC hits `pcg_max_iterations` almost every ADMM step, increase this.
pcg_tol_fraction = 1.0

admm_max_iterations = 2000
pcg_max_iterations = 5

# OSQP-style rho initialization (vectorized rho per constraint).
OSQP_RHO = 1.0
OSQP_RHO_MIN = 1e-6
OSQP_RHO_MAX = 1e6
OSQP_RHO_EQ_OVER_RHO_INEQ = 100
OSQP_RHO_TOL = 0.01

def init_rho_vec(l, u, rho_base=OSQP_RHO):
    rho_base = float(np.clip(rho_base, OSQP_RHO_MIN, OSQP_RHO_MAX))
    rho = np.full(l.shape, rho_base, dtype=np.float32)

    finite_l = np.isfinite(l)
    finite_u = np.isfinite(u)

    loose = (~finite_l) & (~finite_u)  # no bounds
    eq = finite_l & finite_u & ((u - l) < OSQP_RHO_TOL)  # equality (scaled)

    rho[loose] = OSQP_RHO_MIN
    rho[eq] = rho_base * OSQP_RHO_EQ_OVER_RHO_INEQ

    rho = np.clip(rho, OSQP_RHO_MIN, OSQP_RHO_MAX * OSQP_RHO_EQ_OVER_RHO_INEQ).astype(np.float32)

    stats = {
        "rho_base": rho_base,
        "num_loose": int(np.sum(loose)),
        "num_eq": int(np.sum(eq)),
        "num_ineq": int(len(rho) - np.sum(loose) - np.sum(eq)),
        "rho_min": float(np.min(rho)),
        "rho_median": float(np.median(rho)),
        "rho_max": float(np.max(rho)),
    }
    return rho, stats

# IMPORTANT: initialize rho from the *scaled* bounds that are sent to HW.
rho_in, rho_stats = init_rho_vec(l_scaled, u_scaled, rho_base=OSQP_RHO)
print(
    f"Initial rho: base={rho_stats['rho_base']:.3e} | "
    f"eq={rho_stats['num_eq']} ineq={rho_stats['num_ineq']} loose={rho_stats['num_loose']} | "
    f"min/med/max={rho_stats['rho_min']:.3e}/{rho_stats['rho_median']:.3e}/{rho_stats['rho_max']:.3e}"
)

RUN_BOTH_ADAPTIVE_RHO = True

# =====================================================================
# 5. Utility Functions
# =====================================================================

def ceil_div(a, b):
    return (a + b - 1) // b

def pack_csc_to_words(row_idx, values, num_words):
    nnz = len(row_idx)

    row_words = allocate(shape=(num_words, PACK_SIZE), dtype=np.int32, cacheable=False)
    val_words = allocate(shape=(num_words, PACK_SIZE), dtype=np.float32, cacheable=False)
    
    row_words[:] = 0
    val_words[:] = 0.0

    row_words.reshape(-1)[:nnz] = row_idx
    val_words.reshape(-1)[:nnz] = values

    return row_words, val_words

def write_64bit_address(ip, base_offset, address):
    ip.write(base_offset, address & 0xFFFFFFFF)
    ip.write(base_offset + 0x04, (address >> 32) & 0xFFFFFFFF)

def float_to_uint(f_val):
    return struct.unpack('<I', struct.pack('<f', float(f_val)))[0]

def uint_to_float(u_val):
    return struct.unpack('<f', struct.pack('<I', int(u_val)))[0]

def objective_val(x_vec):
    # CRITICAL: Always use the ORIGINAL, unscaled P and q to check the true objective value
    return float(
        0.5 * np.sum(P_diag_orig * x_vec * x_vec)
        + np.dot(q, x_vec)
    )

def max_box_violation(z, l, u):
    return float(
        np.max(np.maximum(0.0, np.maximum(l - z, z - u)))
    )

# =====================================================================
# 6. Allocate Hardware Buffers (USING SCALED DATA)
# =====================================================================

print("\nAllocating hardware buffers...")

A_words = int(ceil_div(A_nnz, PACK_SIZE))
AT_words = int(ceil_div(A_nnz, PACK_SIZE))
P_words = int(ceil_div(P_nnz, PACK_SIZE))

A_row_words, A_val_words = pack_csc_to_words(A_row_idx, A_values, A_words)
AT_row_words, AT_val_words = pack_csc_to_words(AT_row_idx, AT_values, AT_words)
P_row_words, P_val_words = pack_csc_to_words(P_row_idx, P_values, P_words)

PAD = 16

# Pointers
A_col_ptr_hw = allocate(shape=(num_cols + 1 + PAD,), dtype=np.int32, cacheable=False)
A_col_ptr_hw[: num_cols + 1] = A_col_ptr

AT_col_ptr_hw = allocate(shape=(num_rows + 1 + PAD,), dtype=np.int32, cacheable=False)
AT_col_ptr_hw[: num_rows + 1] = AT_col_ptr

P_col_ptr_hw = allocate(shape=(num_cols + 1 + PAD,), dtype=np.int32, cacheable=False)
P_col_ptr_hw[: num_cols + 1] = P_col_ptr

# Dense vectors (SCALED)
P_diag_hw = allocate(shape=(num_cols + PAD,), dtype=np.float32, cacheable=False)
P_diag_hw[:num_cols] = P_values

l_in_hw = allocate(shape=(num_rows + PAD,), dtype=np.float32, cacheable=False)
l_in_hw[:num_rows] = l_scaled

u_in_hw = allocate(shape=(num_rows + PAD,), dtype=np.float32, cacheable=False)
u_in_hw[:num_rows] = u_scaled

q_in_hw = allocate(shape=(num_cols + PAD,), dtype=np.float32, cacheable=False)
q_in_hw[:num_cols] = q_scaled

rho_in_hw = allocate(shape=(num_rows + PAD,), dtype=np.float32, cacheable=False)
rho_in_hw[:num_rows] = rho_in

# Outputs
x_out_hw = allocate(shape=(num_cols + PAD,), dtype=np.float32, cacheable=False)
y_out_hw = allocate(shape=(num_rows + PAD,), dtype=np.float32, cacheable=False)
x_out_hw[:] = 0.0
y_out_hw[:] = 0.0

# =====================================================================
# 7. Hardware Execution
# =====================================================================

def run_hw(adaptive_rho):

    print(f"\n=== HW Run (adaptive_rho={adaptive_rho}) ===")

    # Control registers
    control_ip.write(0x10, num_rows)
    control_ip.write(0x18, num_cols)
    control_ip.write(0x20, A_nnz)
    control_ip.write(0x28, P_nnz)

    control_ip.write(0x30, float_to_uint(sigma))
    control_ip.write(0x38, float_to_uint(alpha))

    control_ip.write(0x40, admm_max_iterations)
    control_ip.write(0x48, pcg_max_iterations)

    control_ip.write(0x50, int(adaptive_rho))

    control_ip.write(0x58, float_to_uint(eps_abs))
    control_ip.write(0x60, float_to_uint(eps_rel))
    control_ip.write(0x68, float_to_uint(pcg_tol_fraction))

    # Address registers
    write_64bit_address(control_r_ip, 0x10, A_row_words.device_address)
    write_64bit_address(control_r_ip, 0x1c, A_col_ptr_hw.device_address)
    write_64bit_address(control_r_ip, 0x28, A_val_words.device_address)

    write_64bit_address(control_r_ip, 0x34, AT_row_words.device_address)
    write_64bit_address(control_r_ip, 0x40, AT_col_ptr_hw.device_address)
    write_64bit_address(control_r_ip, 0x4c, AT_val_words.device_address)

    write_64bit_address(control_r_ip, 0x58, P_row_words.device_address)
    write_64bit_address(control_r_ip, 0x64, P_col_ptr_hw.device_address)
    write_64bit_address(control_r_ip, 0x70, P_val_words.device_address)

    write_64bit_address(control_r_ip, 0x7c, P_diag_hw.device_address)
    write_64bit_address(control_r_ip, 0x88, l_in_hw.device_address)
    write_64bit_address(control_r_ip, 0x94, u_in_hw.device_address)
    write_64bit_address(control_r_ip, 0xa0, q_in_hw.device_address)
    write_64bit_address(control_r_ip, 0xac, rho_in_hw.device_address)
    write_64bit_address(control_r_ip, 0xb8, x_out_hw.device_address)
    write_64bit_address(control_r_ip, 0xc4, y_out_hw.device_address)

    # Start accelerator
    x_out_hw[:] = 0.0
    y_out_hw[:] = 0.0

    hw_start = time.time()
    control_ip.write(0x00, 0x01)

    while (control_ip.read(0x00) & 0x02) == 0:
        pass

    hw_end = time.time()
    print(f"HW execution time: {(hw_end - hw_start) * 1000:.4f} ms")

    # Read results
    admm_iters = int(control_ip.read(0x70))
    pcg_iters = int(control_ip.read(0x80))
    r_prim_out = float(uint_to_float(control_ip.read(0x90)))
    r_dual_out = float(uint_to_float(control_ip.read(0xA0)))
    status_out = int(control_ip.read(0xB0))

    # -------------------------------------------------------------
    # CRITICAL: UNSCALE THE HARDWARE SOLUTION
    # -------------------------------------------------------------
    x_hw_scaled = np.array(x_out_hw[:num_cols], dtype=np.float32)
    x_hw_unscaled = x_hw_scaled * E_scale

    # Compute violations and objective using ORIGINAL matrices
    Ax_hw = (A_sparse @ x_hw_unscaled).astype(np.float32)
    viol_hw = max_box_violation(Ax_hw, l_in, u_in)
    obj_hw = objective_val(x_hw_unscaled)

    # Print summary
    print("\n--- Results ---")
    print(f"Kernel Status: {'Converged' if status_out == 1 else 'Max Iterations'}")
    print(f"ADMM Iterations: {admm_iters}")
    print(f"PCG Iterations : {pcg_iters}")
    if admm_iters > 0:
        print(f"PCG/ADMM       : {pcg_iters / float(admm_iters):.2f} (max={pcg_max_iterations})")
    print(f"Primal Residual: {r_prim_out:.5e}")
    print(f"Dual Residual  : {r_dual_out:.5e}")
    print(f"Constraint Violation: {viol_hw:.3e}")
    print(f"Objective Value: {obj_hw:.6e}")

    return {
        "adaptive_rho": adaptive_rho,
        "status": status_out,
        "admm_iters": admm_iters,
        "pcg_iters": pcg_iters,
        "r_prim": r_prim_out,
        "r_dual": r_dual_out,
        "viol": viol_hw,
        "obj": obj_hw,
        "hw_ms": (hw_end - hw_start) * 1000.0,
    }

# =====================================================================
# 8. Run Tests
# =====================================================================

adaptive_rho_list = [0, 1] if RUN_BOTH_ADAPTIVE_RHO else [0]
results = []
for adaptive_rho in adaptive_rho_list:
    results.append(run_hw(adaptive_rho))

# =====================================================================
# 9. Summary
# =====================================================================

if RUN_BOTH_ADAPTIVE_RHO and len(results) == 2:
    off = results[0]
    on = results[1]
    print("\n=== Summary ===")
    print(f"ADMM iterations: off={off['admm_iters']} | on={on['admm_iters']}")
    print(f"Primal residual: off={off['r_prim']:.3e} | on={on['r_prim']:.3e}")
    print(f"Dual residual: off={off['r_dual']:.3e} | on={on['r_dual']:.3e}")
    print(f"Violation: off={off['viol']:.3e} | on={on['viol']:.3e}")
    print(f"Objective: off={off['obj']:.6e} | on={on['obj']:.6e}")
    print(f"HW time (ms): off={off['hw_ms']:.3f} | on={on['hw_ms']:.3f}")

# =====================================================================
# 10. Cleanup
# =====================================================================

buffers = [
    A_row_words, A_val_words, AT_row_words, AT_val_words,
    P_row_words, P_val_words, A_col_ptr_hw, AT_col_ptr_hw,
    P_col_ptr_hw, P_diag_hw, l_in_hw, u_in_hw, q_in_hw,
    rho_in_hw, x_out_hw, y_out_hw
]

for buf in buffers:
    buf.freebuffer()

print("\nBuffers released.")

Tuning:

alpha = 1.8
sigma = 1e-2

eps_abs = 1e-3
eps_rel = 1e-3

Indirect solve accuracy control:
If MPC hits `pcg_max_iterations` almost every ADMM step, increase this.
pcg_tol_fraction = 1.0

admm_max_iterations = 2000
pcg_max_iterations = 5

OSQP-style rho initialization (vectorized rho per constraint).
OSQP_RHO = 1.0
OSQP_RHO_MIN = 1e-6
OSQP_RHO_MAX = 1e6
OSQP_RHO_EQ_OVER_RHO_INEQ = 100
OSQP_RHO_TOL = 0.01